#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,DateType
from pyspark.sql.functions import trim,col,length

#Reading From Bronze Layer

In [0]:
df=spark.table("workspace.bronze.erp_loc_a101")

#Data Transformation

#Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

#Normalization

## Customer Id Cleanup

In [0]:
df = df.withColumn(
    "cid",
    F.regexp_replace(col("cid"),"-",""))

## Country Cleaning

In [0]:
df= df.withColumn(
    "cntry",
    F.when(F.upper(col('cntry'))=='DE',"Germany")
    .when(F.upper(col('cntry')).isin("US","USA"),"United States")
    .when((col('cntry')=="")|(col('cntry').isNull()),"n/a")
    .otherwise(col('cntry'))
)

#Renaming columns

In [0]:

RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)


## Sanity checks of dataframe

In [0]:
display(df.limit(10))

#Write to Silver Layer

In [0]:
(
df.write
.mode("overwrite")
.format('delta')
.saveAsTable("silver.erp_customer_location")
)